[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/html_listing_scraper.ipynb)

# Scraping a listings page with BeautifulSoup

A local-services directory lists businesses as cards: name, locality, rating, services offered, response time. We download a saved copy of one such page, parse it with BeautifulSoup and turn each card into a row of a CSV.

## Setup

Install the parser if you don't already have it (uncomment the line below).

In [ ]:
# !pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from copy import copy

## Download the page and save it locally

The page is a saved copy hosted with the course material, so it works the same on Colab and on your laptop. Saving the HTML once means we can re-run the parsing cells without downloading again.

In [ ]:
url = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20collection/data/service_listings.html"
response = requests.get(url)
print(response.status_code)
if response.status_code == 200:
    with open("service_listings.html", "w", encoding="utf-8") as f:
        f.write(response.text)

## Parse the saved HTML

Open `service_listings.html` in a browser and *View Source* to see the structure we are about to walk.

In [ ]:
with open('service_listings.html', 'r', encoding='utf-8') as file:
    html_content = file.read()
soup = bs(html_content, 'lxml')  # html.parser, lxml, html5lib

## Extract fields from each business card

For every `div.sk-card`: id, name, locality, lat/long, rating, list of services, response time and score.

In [ ]:
cards = soup.find_all('div', class_="sk-card")  # this gets the whole div
print(len(cards), "cards")
sulekha_data=[]
business_info={}
for card in cards:
    # Id, name
    business_info['id']=card.get('businessid',"")
    business_info['name']=card.get('businessname',"")
    
    print("businessid",card.get('businessid',""))
    print("business name",card.get('businessname',""))
    #locality
    locality = card.find('div',class_="locality")
    #print(locality)
    location = locality.find('span')
    business_info['locality']=location.get_text()
    location=locality.find('div',class_='sk-link')
    if location:
        business_info['lat']=location.get('businesslat',"")
        business_info['long']=location.get('businesslong',"")
    else:
        business_info['lat']=''
        business_info['long']=''
    #Ratings
    rating_div=card.find('div','ratings-group')
    rating=rating_div.find('b')
    if rating:
        business_info['rating']=rating.get_text()
    else:
        business_info['rating']=''
    
    #list of services
    services_div=card.find('div',class_="tags-link")
    
    services_list=services_div.find_all('spam',class_="tag-link-item")
    services=[]
    if not services_list:
        services_list=services_div.find_all('span',class_="tag-link-item") 
    for service in services_list:
        service_name=service.get_text()
        if service_name not in services:
            services.append(service_name)
    
    #print(services_list)
    business_info['services']=services

    #keypoints
    key_points=card.find('div','key-points light')
    #print(key_points)
    key_point_span = key_points.find_all('span')
    for key_point in key_point_span:
        if 'Response Time' in key_point.get_text():
            b_tag=key_point.find('b')
            business_info['response_time']=b_tag.get_text()
        if 'Sulekha score' in key_point.get_text():
            b_tag=key_point.find('b')
            business_info['sulekha_score']=b_tag.get_text()


    sulekha_data.append(copy(business_info))
    #print(business_info)
    #exit()

## Save to CSV

In [ ]:
df = pd.DataFrame(sulekha_data)
df.to_csv('service_listings.csv', index=None)
df